# ⚡ TwinIQ — AI-Powered Product Intelligence for Industrial Commerce
## Electric Motors

### Problem Statement
Manufacturers publish product information across websites, catalogues and technical documents. Converting fragmented information into trusted, structured, commerce-ready product intelligence is difficult to scale.

### TwinIQ Task
1. Start with minimal product information such as **Brand + Manufacturer Part Number + Short Description**.
2. Collect supporting public product information and technical documents.
3. Extract and normalize technical specifications.
4. Validate data and keep source evidence.
5. Build a structured Electric Motor product record.
6. Accept a natural-language requirement.
7. Explain whether the product is **Suitable, Partially Suitable, or Not Suitable**.

This notebook follows the same week-by-week learning style as the supplied DSV notebook: loading → understanding → cleaning → analysis → feature engineering → model learning → dimensionality reduction → final system.

In [ ]:
import os, re, json, warnings, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
# Make the project root available when the notebook is opened from the notebooks folder.
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


---
# Week 1 — Define the Product Intelligence Schema

Before collecting data, we define exactly what TwinIQ must learn about an Electric Motor. This prevents random data collection and ensures every source is converted into the same canonical structure.

In [ ]:
MOTOR_SCHEMA = {
    "identity": ["brand","manufacturer_part_number","series","description"],
    "technical": ["motor_type","rated_power_kw","voltage_v","frequency_hz","speed_rpm",
                  "current_a","efficiency_percent","power_factor","ip_rating",
                  "insulation_class","mounting","duty","frame_size","ambient_temperature_c",
                  "weight_kg","poles"],
    "traceability": ["source_type","source_reference"]
}
MOTOR_SCHEMA

---
# Week 2 — Know and Collect Your Dataset
> **Focus:** Dataset collection, loading and basic exploration

TwinIQ does not depend on one Kaggle dataset. The real project dataset is created from **official manufacturer product records and datasheets**. We therefore keep a source registry (`data/sources.csv`) and a local structured dataset.

For immediate development, this package includes a starter seed dataset so every notebook cell can run. Before final submission, replace or extend those records with real evidence-backed records.

In [ ]:
DATA_PATH = Path("../data/raw/electric_motors_seed.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/electric_motors_seed.csv")
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df.info()

In [ ]:
print('Rows and Columns:', df.shape)
print('\nMissing Values:\n', df.isnull().sum())

In [ ]:
df.describe(include='all').T

## Dataset Source Registry
To collect real data, open `data/sources.csv` and enter official product or datasheet URLs. The collection function below downloads only URLs you explicitly provide.

In [ ]:
sources_path = Path("../data/sources.csv")
if not sources_path.exists():
    sources_path = Path("data/sources.csv")
sources = pd.read_csv(sources_path)
sources.head()

### Week 2 Outcome
- Dataset loaded successfully.
- Product schema understood.
- Missing values and data types inspected.
- Source registry prepared for official evidence collection.

---
# Week 3 — The Cleaning Sprint

The goal is to make product records consistent and machine-readable. We:
1. Remove duplicate Brand + MPN combinations.
2. Convert numeric specifications to numbers.
3. Standardize IP ratings and duty codes.
4. Derive poles when needed.
5. Preserve missing values instead of inventing technical specifications.

In [ ]:
from src.clean import clean_motor_data
df_clean = clean_motor_data(df)
print('Before:', df.shape)
print('After :', df_clean.shape)
df_clean.head()

In [ ]:
print('Duplicate product identities:', df_clean.duplicated(subset=['brand','manufacturer_part_number']).sum())
print('\nMissing values after cleaning:\n', df_clean.isnull().sum())

In [ ]:
num_cols = df_clean.select_dtypes(include=np.number).columns
plt.figure(figsize=(14,6))
df_clean[num_cols].boxplot(rot=45)
plt.title('Numerical Specification Outlier Inspection')
plt.show()

### Validation Logic
Extreme values are not automatically deleted. Industrial motors can legitimately have very different ratings. Instead, TwinIQ flags impossible or suspicious values for review.

In [ ]:
def validation_flags(row):
    flags=[]
    if pd.notna(row['rated_power_kw']) and row['rated_power_kw'] <= 0: flags.append('Invalid power')
    if pd.notna(row['frequency_hz']) and row['frequency_hz'] not in [50,60]: flags.append('Unusual frequency')
    if pd.notna(row['efficiency_percent']) and not (0 < row['efficiency_percent'] <= 100): flags.append('Invalid efficiency')
    if pd.notna(row['power_factor']) and not (0 < row['power_factor'] <= 1): flags.append('Invalid power factor')
    if pd.notna(row['speed_rpm']) and row['speed_rpm'] <= 0: flags.append('Invalid speed')
    return flags

df_clean['validation_flags'] = df_clean.apply(validation_flags, axis=1)
df_clean[['manufacturer_part_number','validation_flags']].head()

---
# Week 4 — Exploratory Data Analysis (EDA)

We learn how the motor dataset behaves before building any model.

In [ ]:
plt.figure(figsize=(8,4))
sns.countplot(data=df_clean, x='brand')
plt.title('Motor Records by Brand')
plt.xticks(rotation=20)
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df_clean['rated_power_kw'], bins=20, kde=True)
plt.title('Rated Power Distribution (kW)')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df_clean, x='rated_power_kw', y='efficiency_percent', hue='brand')
plt.title('Power vs Efficiency')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df_clean, x='ip_rating', y='rated_power_kw')
plt.title('Rated Power by IP Rating')
plt.show()

In [ ]:
numeric_for_corr = ['rated_power_kw','voltage_v','frequency_hz','speed_rpm',
                   'current_a','efficiency_percent','power_factor','weight_kg','poles']
corr = df_clean[numeric_for_corr].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt='.2f', center=0)
plt.title('Electric Motor Numerical Feature Correlation')
plt.show()

### Week 4 Insights
Use the plots to identify distribution, relationships and possible redundancy. Do not assume a relationship is causal only because a chart shows correlation.

---
# Week 5 — Feature Engineering

We convert raw technical specifications into additional meaningful features while retaining the original specifications for traceability.

In [ ]:
from src.features import add_features
df_feat = add_features(df_clean)
df_feat[['rated_power_kw','power_class','speed_rpm','speed_band',
         'efficiency_percent','efficiency_band']].head()

In [ ]:
print('Feature Types:')
for col in df_feat.columns:
    print(f'{col}: {df_feat[col].dtype}')

## Encoding for Learning
Machine-learning models need numeric inputs. We use one-hot encoding for low-cardinality categorical fields and preserve the original dataset for human-readable product intelligence.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

numeric_features = ['rated_power_kw','voltage_v','frequency_hz','speed_rpm',
                    'current_a','efficiency_percent','power_factor','weight_kg','poles']
categorical_features = ['brand','ip_rating','mounting','duty','insulation_class']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

---
# Week 6 — Dataset Learning: Compatibility Model

**Important design decision:** The final suitability verdict is deterministic and explainable; an LLM is not allowed to silently decide engineering compatibility.

For dataset learning, we create training examples from product requirements. Each example is labelled `1` when the requirement matches the product within defined tolerances and `0` otherwise. The trained model provides an additional learned compatibility signal, while the rule-based engine remains the final explainable decision layer.

In [ ]:
from src.match import match_product
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

rng = np.random.default_rng(42)
train_rows=[]

for _, product in df_feat.iterrows():
    product_dict = product.to_dict()
    # Positive requirement: close to product specification
    req_pos = {
        'rated_power_kw': float(product['rated_power_kw'] * rng.uniform(0.97,1.03)),
        'voltage_v': float(product['voltage_v']),
        'frequency_hz': float(product['frequency_hz']),
        'speed_rpm': float(product['speed_rpm'] * rng.uniform(0.97,1.03)),
        'ip_rating': product['ip_rating'],
        'duty': product['duty']
    }
    # Negative requirement: intentionally mismatched power/voltage
    req_neg = req_pos.copy()
    req_neg['rated_power_kw'] = float(product['rated_power_kw'] * rng.choice([0.5, 1.8, 2.5]))
    req_neg['voltage_v'] = float(rng.choice([230, 380, 415, 690]))
    for req, label in [(req_pos,1),(req_neg,0)]:
        train_rows.append({
            'product_power':product['rated_power_kw'],
            'product_voltage':product['voltage_v'],
            'product_frequency':product['frequency_hz'],
            'product_speed':product['speed_rpm'],
            'product_efficiency':product['efficiency_percent'],
            'req_power':req['rated_power_kw'],
            'req_voltage':req['voltage_v'],
            'req_frequency':req['frequency_hz'],
            'req_speed':req['speed_rpm'],
            'power_diff':abs(product['rated_power_kw']-req['rated_power_kw']),
            'voltage_diff':abs(product['voltage_v']-req['voltage_v']),
            'speed_diff':abs(product['speed_rpm']-req['speed_rpm']),
            'label':label
        })
compat_df = pd.DataFrame(train_rows)
compat_df.head()

In [ ]:
X = compat_df.drop(columns='label')
y = compat_df['label']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)

rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf.fit(X_train,y_train)
pred = rf.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test,pred),4))
print('\nClassification Report:\n', classification_report(y_test,pred))

In [ ]:
cm = confusion_matrix(y_test,pred)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['Not Match','Match'], yticklabels=['Not Match','Match'])
plt.title('Compatibility Model Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(8,5))
importance.plot(kind='bar')
plt.title('Compatibility Model Feature Importance')
plt.ylabel('Importance')
plt.show()

### Week 6 Result
The model learns patterns from labelled product-requirement examples. In the final application:
- **Rule engine:** final transparent PASS/FAIL checks.
- **ML model:** supplementary learned compatibility signal.
- **AI/LLM:** extraction and explanation, never silent engineering arithmetic.

---
# Week 7 — Dimensionality Reduction

This section follows the supplied notebook style: correlation analysis → VIF → variance thresholding → PCA.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA

X_dim = df_feat[numeric_features].dropna().copy()
X_dim = X_dim.replace([np.inf,-np.inf],np.nan).dropna()

corr_dim = X_dim.corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr_dim, annot=True, fmt='.2f', center=0)
plt.title('Correlation Analysis Before PCA')
plt.show()

In [ ]:
def calculate_vif(X_data):
    vif_df = pd.DataFrame()
    vif_df['Feature'] = X_data.columns
    vif_df['VIF_Score'] = [variance_inflation_factor(X_data.values, i)
                           for i in range(X_data.shape[1])]
    return vif_df.sort_values('VIF_Score', ascending=False)

vif = calculate_vif(X_dim)
vif

In [ ]:
vt = VarianceThreshold(threshold=0.0)
X_vt = vt.fit_transform(X_dim)
kept_features = X_dim.columns[vt.get_support()].tolist()
print('Features kept after variance thresholding:', kept_features)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_dim[kept_features])

pca_full = PCA().fit(X_scaled)
exp_var = pca_full.explained_variance_ratio_
cum_var = np.cumsum(exp_var)
n95 = int(np.argmax(cum_var >= 0.95) + 1)

print('Components needed for 95% variance:', n95)

plt.figure(figsize=(9,4))
plt.plot(range(1,len(cum_var)+1), cum_var*100, marker='o')
plt.axhline(95, linestyle='--')
plt.axvline(n95, linestyle='--')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance (%)')
plt.title('PCA Cumulative Variance')
plt.show()

In [ ]:
pca = PCA(n_components=n95)
X_pca = pca.fit_transform(X_scaled)
df_pca = pd.DataFrame(X_pca, columns=[f'PC{i}' for i in range(1,n95+1)])
df_pca.head()

### Week 7 Conclusion
PCA is used here for exploratory dimensionality reduction. The production requirement-matching engine should still retain original technical specifications so the user can see exact values and evidence.

---
# Final Stage — Build Structured Product Intelligence

Each cleaned motor row is converted into a structured record that can be stored as JSON and shown in the catalog.

In [ ]:
def row_to_product(row):
    technical_cols = ['motor_type','rated_power_kw','voltage_v','frequency_hz','speed_rpm',
                      'current_a','efficiency_percent','power_factor','ip_rating',
                      'insulation_class','mounting','duty','frame_size',
                      'ambient_temperature_c','weight_kg','poles']
    return {
        'product': {
            'brand': row['brand'],
            'manufacturer_part_number': row['manufacturer_part_number'],
            'series': row.get('series'),
            'description': row.get('description')
        },
        'technical_specifications': {
            c: (None if pd.isna(row.get(c)) else row.get(c)) for c in technical_cols
        },
        'traceability': {
            'source_type': row.get('source_type'),
            'source_reference': row.get('source_reference')
        }
    }

example_product = row_to_product(df_feat.iloc[0])
example_product

In [ ]:
from src.storage import save_product_json
saved = save_product_json(example_product, '../data/structured' if Path('../data').exists() else 'data/structured')
print('Saved:', saved)

---
# Final Stage — Requirement Match (Core TwinIQ Feature)

The user gives a natural-language requirement. TwinIQ:
1. Parses the requirement.
2. Compares it with the selected product.
3. Produces field-level PASS / FAIL / UNKNOWN.
4. Calculates an explainable score.
5. Returns a verdict.

In [ ]:
from src.match import parse_requirement, match_product

requirement_text = '''
I need a three-phase industrial electric motor with 15 kW power,
415 V voltage, 50 Hz frequency, approximately 1500 RPM speed,
IP55 protection and suitable for continuous operation.
'''

parsed_requirement = parse_requirement(requirement_text)
parsed_requirement

In [ ]:
# Choose a product from the dataset
selected_product = df_feat.iloc[0].to_dict()
result = match_product(selected_product, parsed_requirement)

print('Verdict:', result['verdict'])
print('Score:', result['score'])
pd.DataFrame(result['checks'])

---
# Final Project Outcome

## Complete TwinIQ Pipeline

Minimal Product Input
↓
Official Source Registry / Technical Datasheets
↓
PDF Text Extraction
↓
Cleaning + Normalization
↓
Validation + Traceability
↓
EDA + Dataset Learning
↓
Feature Engineering
↓
Optional ML Compatibility Signal
↓
Structured Product Intelligence
↓
Natural-Language Requirement
↓
Explainable Rule-Based Match
↓
**SUITABLE / PARTIALLY SUITABLE / NOT SUITABLE**

## Production Files
- Run the Streamlit UI with: `streamlit run app.py`
- Add real source URLs in `data/sources.csv`
- Download public datasheets with `src.collect.download_from_sources()`
- Extract page-wise evidence with `src.pdf_extract.extract_pdf_text()`
- Keep the original requirement unchanged: structured product intelligence + explainable suitability.

---
# PDF-First Product Intelligence

This section adds direct PDF analysis. Instead of manually entering product text, upload an electric-motor datasheet, extract page text, convert available specifications into structured fields, and preserve page-level evidence. Missing values remain missing; the extraction rules do not invent specifications.


In [ ]:
from pathlib import Path
import sys

if Path('../src').exists():
    sys.path.append(str(Path('..').resolve()))

from src.pdf_extract import analyze_motor_pdf

PDF_PATH = Path('../data/raw/your_motor_datasheet.pdf')
if not PDF_PATH.exists():
    print('Add a motor datasheet PDF at data/raw/your_motor_datasheet.pdf, then run this cell again.')
else:
    pdf_result = analyze_motor_pdf(PDF_PATH, PDF_PATH.name)
    pdf_product = pdf_result['product']
    print('Pages extracted:', len(pdf_result['pages']))
    display(pd.DataFrame([pdf_product]))


In [ ]:
# Show page-level evidence for the PDF extraction
if 'pdf_result' in globals():
    evidence_rows = []
    for field, ev in pdf_result['evidence'].items():
        evidence_rows.append({'field': field, 'page': ev['page'], 'evidence': ev['snippet']})
    display(pd.DataFrame(evidence_rows))


### PDF Upload in the Final App

The Streamlit application now has a **PDF Analysis** tab. The final demo flow is:

`Upload PDF → Analyze PDF → Structured Product Intelligence → Requirement Match → Explainable Result`

This means the user can analyze a datasheet directly instead of manually typing the motor specifications.


# Final Stage — Recommendation Catalog
Rank all motors for a user requirement and return the best alternatives with explainable reasons.

In [ ]:
from src.match import parse_requirement, recommend_products

user_need = "I need a three-phase industrial motor with 15 kW power, 415 V, 50 Hz, around 1500 RPM, IP55 and continuous operation."
req = parse_requirement(user_need)
recommendations = recommend_products(df, req, top_n=5)

rows = []
for rank, item in enumerate(recommendations, start=1):
    p = item["product"]
    r = item["result"]
    rows.append({
        "Rank": rank,
        "Part Number": p.get("manufacturer_part_number"),
        "Brand": p.get("brand"),
        "Power (kW)": p.get("rated_power_kw"),
        "Compatibility (%)": r["score"],
        "Verdict": r["verdict"]
    })
pd.DataFrame(rows)
